# Alarm pheromone imaging: Hart, Frank, Lopes et al. (2023), dandiset 000467

Paper: Hart T, Frank DD, Lopes LE et al. "Sparse and stereotyped encoding implicates a core glomerulus
for ant alarm behavior", Cell 186 (2023), doi:10.1016/j.cell.2023.05.025.

Dataset: DANDI:000467, version 0.261004.1655 (doi:10.48324/dandi.000467/0.261004.1655). The paper
cites the dandiset without a version. This is the only published version (published 2026-10-04,
"after adding additional meta data"); the draft has the same 14,685 assets and size, so the most
recent published version is pinned, as the protocol says when the paper cites none.

Claims file: `claims/000467/10.1016-j.cell.2023.05.025.yaml`, written from the accepted author
manuscript in PubMed Central (PMC10334690) with its supplement. The shared methods M1 to M4 are in
its opening comment; the helper `al_imaging.py` says where each parameter comes from.

This notebook holds the unilateral alarm pheromone experiment (13 ants). The general odorant and
bilateral experiments are in `general_odorants.ipynb`. Claims tested here:

| Claim | Here |
|---|---|
| fig4b-vehicle-no-response | tested |
| figs5a-sparse-median-6 | tested |
| figs5b-shared-glomerulus | tested |
| figs5c-mho-rarely-overlaps | tested |
| fig6-shared-panic-pathway | imaging half tested |
| fig4-no-decreases | tested at each ant's higher concentration |
| fig4-large-lasting-responses | tested |
| fig5ab-focal-glomeruli | tested |
| fig5d-stereotyped-positions | tested |
| fig5c-s6-tuning | tested |
| fig5c-6g-slow | tested |
| figs7a-6g-extended | tested |
| figs7c-temporal-lme | tested |
| figs5a-concentration-more-glomeruli | in `concentration.ipynb` |
| fig3f, fig3fg, figs4a | in `general_odorants.ipynb` |
| behaviour, transgenesis, confocal and genome claims (15) | blocked; see the attempt record |

Whole-volume peak maps (M1) are computed for each ant's higher-concentration pheromone trials,
its vehicle trial and its isopropanol trial: 182 trials of 33 planes, about 6,000 files of 84 MB,
of which only volumes 1 to 5, 8 and 9 to 14 are read. The single-glomerulus traces (M3) read every
volume of a band of rows around the focal glomeruli, on five planes, for every pheromone trial.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import ndimage, stats

import al_imaging as al

from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # convergence is reported with each fit
warnings.filterwarnings("ignore", message="Random effects covariance is singular")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
T0 = time.time()
print("dandiset", al.DANDISET_ID, "version", al.DANDISET_VERSION)

In [ ]:
assets = al.list_assets()
trials = al.trials(assets)
print(len(assets), "assets;", assets.blob.nunique(), "distinct blobs; all file names parsed:", assets.parsed.all())
alarm_ants = sorted(trials[trials.odor == "heptanone"].ant.unique())
general_ants = sorted(trials[trials.odor == "3-hexanone"].ant.unique())
bilateral_ants = sorted(set(trials.ant) - set(alarm_ants) - set(general_ants))
print("alarm ants", len(alarm_ants), "general odorant ants", len(general_ants), "bilateral ants", len(bilateral_ants), bilateral_ants)
alarm = trials[trials.ant.isin(alarm_ants)].copy()
conc_pairs = alarm[alarm.odor.isin(al.PHEROMONES)].groupby("ant").conc.agg(lambda c: tuple(sorted(set(c))))
alarm["higher"] = alarm.ant.map(conc_pairs.map(max))
print(conc_pairs.value_counts())

In [ ]:
print("Trials per ant, odor and concentration (alarm experiment; planes per trial:",
      sorted(alarm.n_planes.unique()), "; volumes per trial:", sorted(alarm.nvol.unique()), ")")
print(alarm.groupby(["ant", "conc", "odor"]).size().unstack("odor").fillna(0).astype(int).to_string())
print()
print("Recordings of 120 volumes (144 s):")
print(alarm[alarm.nvol == 120][["ant", "trial", "odor", "conc", "n_planes"]].to_string(index=False))
# Each ant's sessions: date and ant number in the name; one subject per ant
print("subjects:", assets[assets.ant.isin(alarm_ants)]["sub"].nunique(), "; sessions (date):",
      assets[assets.ant.isin(alarm_ants)].date.value_counts().to_dict())

In [ ]:
# Metadata from the files themselves: one file per ant, plus all 33 planes of one trial and one 144 s file
meta_rows = []
for ant in alarm_ants:
    r = alarm[(alarm.ant == ant) & (alarm.nvol == 40)].iloc[0]
    m = al.file_metadata(r.blobs[16])
    meta_rows.append({"ant": ant, **{k: m[k] for k in ("shape", "dtype", "chunks", "compression", "offset", "unit",
                                                       "imaging_rate", "grid_spacing", "age", "location")}})
meta = pd.DataFrame(meta_rows)
print(meta.to_string(index=False))
r = alarm[alarm.nvol == 120].iloc[0]
m120 = al.file_metadata(r.blobs[0])
print("144 s file:", m120["shape"], "offset", m120["offset"], "|", m120["series_description"])
print("experiment description of one file:", al.file_metadata(alarm.iloc[0].blobs[0])["experiment_description"])
one = alarm[(alarm.nvol == 40)].iloc[3]
starts = [al.file_metadata(b)["starting_time"] for b in one.blobs]
print("starting_time by plane (s), trial", one.ant, one.trial, ":", np.round(starts, 4).tolist())
print("step between planes", np.round(np.diff(starts).mean(), 5), "s; x 33 =", np.round(np.diff(starts).mean() * 33, 4),
      "s per volume; 1/imaging_rate =", round(1 / 0.83, 4))
PLANE_DT = float(np.diff(starts).mean())   # each plane's volumes start PLANE_DT x plane later
# the raw range reads equal the h5py reads
with al.open_nwb(one.blobs[10]) as f:
    h = f["acquisition/TwoPhotonSeries1/data"][8:10, 100:110, :]
print("range read equals h5py read:", np.array_equal(al.read_frames(one.blobs[10], 8, 10)[:, 100:110, :], h))
print("byte offset is", sorted(set(meta.offset) | {m120["offset"]}), "in every file read; file sizes:",
      assets.groupby("nvol")["size"].agg(["min", "max"]).to_dict("index"))

The files give the orientation ("right antennal lobe, up is lateral, right is anterior"), so image
rows run lateral (top) to medial and columns posterior (left) to anterior. Pixel size 0.2889 um and 5
um between planes, as the paper states. Ages are given as ranges ("P97D/P104D"). The plane's
`starting_time` grows by about 1.2 s / 33 per plane, so each file's volumes are timed from its plane's
place in the volume scan.

In [ ]:
# Whole-volume peak maps (M1) for each ant's higher concentration, vehicle and isopropanol
sel = alarm[(alarm.nvol == 40) & ((alarm.conc == alarm.higher) | (alarm.odor.isin(["vehicle", "isopropanol"])))]
sel = sel[sel.odor.isin(al.PHEROMONES + ["vehicle", "isopropanol"])]
print(len(sel), "trials,", sum(len(b) for b in sel.blobs), "plane files")
t = time.time()
maps = al.trial_maps(sel, keep_frames=True, progress=lambda a, b: print(f"{a}/{b} files, {time.time() - t:.0f} s", flush=True) if a % 2000 == 0 else None)
print(f"maps in {time.time() - t:.0f} s")

In [ ]:
shift = pd.Series({i: maps[i]["shift"].max() for i in sel.index})
sat = pd.Series({i: maps[i]["saturated"].mean() for i in sel.index})
f0max = pd.Series({i: maps[i]["f0"].max() for i in sel.index})
print("largest frame shift per trial (pixels): median", shift.median(), "max", shift.max(),
      "; trials with a shift over 10 pixels (2.9 um):", int((shift > 10).sum()))
print("fraction of baseline pixels at the detector maximum (8191), mean over planes: median",
      round(sat.median(), 4), "max", round(sat.max(), 4))
print("baseline F0 (4x4 bins) maximum per trial: range", f0max.min().round(0), "to", f0max.max().round(0))
proj_rng = pd.Series({i: (maps[i]["proj"].min(), maps[i]["proj"].max()) for i in sel.index})
print("max projections are clipped to [-1, 4]: min", min(p[0] for p in proj_rng), "max", max(p[1] for p in proj_rng))
# duplicate recordings: identical baseline maps between two trials of an ant
dups = []
for ant, g in sel.groupby("ant"):
    idx = list(g.index)
    for a in range(len(idx)):
        for b in range(a + 1, len(idx)):
            if np.array_equal(maps[idx[a]]["f0"], maps[idx[b]]["f0"]):
                dups.append((ant, sel.loc[idx[a]].trial, sel.loc[idx[b]].trial))
print("pairs of trials with identical baseline volumes:", dups if dups else "none")

## ROIs and responding glomeruli (M2)

For each ant, ROIs are drawn on its higher-concentration pheromone trials (the vehicle and
isopropanol have one trial each, so they cannot meet "at least two trials"), inside a mask of the
AL: the connected regions of at least 0.2 in each trial's map that recur (overlap by at least half
the smaller region) in another trial of the same odor, drawing a keeping all of them and drawing b
those of glomerulus size (10 to 20 um, wider ones split). Each claim built on M2 is shown under the
eight readings: ROI drawing a or b, ROI value i (mean) or ii (maximum), counting x (trial mean at
least 0.2) or y (mean of per-trial counts). The figure shows every ant's ROIs under both drawings.

In [ ]:
al_masks = {ant: al.al_mask(maps, sel[sel.ant == ant].index) for ant in alarm_ants}
rois = {}
for ant in alarm_ants:
    hi = sel[(sel.ant == ant) & sel.odor.isin(al.PHEROMONES)]
    for d in "ab":
        rois[ant, d] = al.draw_rois(maps, hi, d, al_masks[ant])
print(pd.DataFrame({d: {ant: f"{len(rois[ant, d])} regions / {al.n_groups(rois[ant, d])} ROIs" for ant in alarm_ants} for d in "ab"}))

ODORS = al.PHEROMONES + ["vehicle", "isopropanol"]
resp, counts, depth = {}, [], {}
for ant in alarm_ants:
    g = sel[sel.ant == ant]
    for d, v, c in al.READINGS:
        for od in ODORS:
            idx = g[g.odor == od].index
            r_, n_ = al.responding(maps, rois[ant, d], idx, v, c)
            resp[ant, d, v, c, od] = r_
            counts.append({"ant": ant, "higher": g.conc.max() if od in al.PHEROMONES else g[g.odor.isin(al.PHEROMONES)].conc.max(),
                           "reading": d + v + c, "odor": od, "count": n_})
    for d in "ab":
        for od in al.PHEROMONES:
            depth[ant, d, od] = al.group_depths(maps, rois[ant, d], g[g.odor == od].index)
counts = pd.DataFrame(counts)
counts["higher"] = counts.ant.map(conc_pairs.map(max))
wide = counts.pivot_table(index=["ant", "higher"], columns=["reading", "odor"], values="count")
print(wide.round(1).to_string())

# ROI sizes and an image of the ROIs of each ant over its max projection (maximum over its
# higher-concentration pheromone trials), for both drawings
size_rows = []
for (ant, d), rr in rois.items():
    for r_ in rr:
        size_rows.append({"drawing": d, "diameter_um": 2 * np.sqrt(r_["mask"].sum() / np.pi) * al.PIXEL_UM * al.BIN})
size_rows = pd.DataFrame(size_rows)
print(size_rows.groupby("drawing").diameter_um.describe(percentiles=[0.1, 0.5, 0.9]).round(1).to_string())
print("regions under 10 um across, by drawing:", size_rows[size_rows.diameter_um < 10].groupby("drawing").size().to_dict())
fig, ax = plt.subplots(2, len(alarm_ants), figsize=(1.9 * len(alarm_ants), 4.2), squeeze=False)
for k, ant in enumerate(alarm_ants):
    hi = sel[(sel.ant == ant) & sel.odor.isin(al.PHEROMONES)]
    bg = np.max([al._bin(maps[i]["proj"], al.BIN) for i in hi.index], axis=0)
    for j, d in enumerate("ab"):
        rgb = np.repeat(((np.clip(bg, -0.2, 1) + 0.2) / 1.2)[..., None], 3, 2)
        edge = np.zeros(bg.shape, bool)
        for m in al.group_masks(rois[ant, d]):
            edge |= m & ~ndimage.binary_erosion(m)
        rgb[edge] = (1, 0, 0)
        rgb[al_masks[ant] & ~ndimage.binary_erosion(al_masks[ant])] = (0, 0.8, 0.8)
        ax[j, k].imshow(rgb); ax[j, k].axis("off")
        ax[j, k].set_title(f"{ant.split('-')[0]}-{ant.split('-')[1]}-{ant.split('-')[2]} {d}: {al.n_groups(rois[ant, d])}", fontsize=6)
plt.suptitle("ROI outlines (red) and AL mask (cyan) over the max projection, clipped to [-0.2, 1]; top drawing a, bottom b", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Variant used below: leave out trials where most of the AL is at least 0.2 (a non-specific change,
# as with the "large motion artifacts" the Methods excluded without giving a rule)
frac_al = pd.Series({i: (al._bin(maps[i]["proj"], al.BIN)[al_masks[sel.loc[i].ant]] >= al.THRESHOLD).mean() for i in sel.index})
print("fraction of AL bins at least 0.2 per trial: median", frac_al.median().round(3), "max", frac_al.max().round(3))
flagged = sel.loc[frac_al[frac_al > 0.5].index, ["ant", "trial", "odor", "conc"]]
print("trials with more than half the AL at least 0.2:", len(flagged))
print(flagged.to_string(index=False))
veh_idx = sel[sel.odor == "vehicle"].index
print("vehicle trials: fraction of AL bins at least 0.2, and the largest frame shift (pixels):")
print(pd.DataFrame({"ant": sel.loc[veh_idx].ant, "AL fraction >= 0.2": frac_al[veh_idx].round(3),
                    "largest shift": [maps[i]["shift"].max() for i in veh_idx]}).to_string(index=False))

counts_x = []
for ant in alarm_ants:
    g = sel[(sel.ant == ant) & ~sel.index.isin(flagged.index)]
    for d, v, c in al.READINGS:
        for od in al.PHEROMONES:
            idx = g[g.odor == od].index
            counts_x.append({"ant": ant, "reading": d + v + c, "odor": od,
                             "count": al.responding(maps, rois[ant, d], idx, v, c)[1] if len(idx) else np.nan})
counts_x = pd.DataFrame(counts_x)
counts_x["higher"] = counts_x.ant.map(conc_pairs.map(max))

## fig4b-vehicle-no-response

> Sparse, unique subsets of glomeruli respond to each alarm pheromone, while the paraffin oil vehicle
> generates no responses.

Criterion: no ROI responds in any ant's vehicle trial, and each pheromone has at least one
responding ROI in most ants at their higher concentration, under every M2 reading; the largest
vehicle ROI value is reported.

In [ ]:
rows = []
for d, v, c in al.READINGS:
    k = d + v + c
    sub = counts[counts.reading == k]
    veh = sub[sub.odor == "vehicle"]
    vmax = []
    for ant in alarm_ants:
        vi = sel[(sel.ant == ant) & (sel.odor == "vehicle")].index[0]
        vals = al.roi_values(maps, rois[ant, d], vi)[0 if v == "i" else 1]
        vmax.append(vals.max() if len(vals) else np.nan)
    row = {"reading": k, "ants with a vehicle-responding ROI": int((veh["count"] > 0).sum()),
           "largest vehicle ROI value": round(float(np.nanmax(vmax)), 3)}
    for od in al.PHEROMONES:
        row[f"ants with >=1 {od} ROI"] = int((sub[sub.odor == od]["count"] > 0).sum())
    rows.append(row)
fig4b = pd.DataFrame(rows)
print("of", len(alarm_ants), "ants")
print(fig4b.to_string(index=False))
ok = (fig4b["ants with a vehicle-responding ROI"] == 0) & np.all([fig4b[f"ants with >=1 {od} ROI"] > len(alarm_ants) / 2 for od in al.PHEROMONES], axis=0)
print("criterion met under readings:", list(fig4b.reading[ok]), "; not met under:", list(fig4b.reading[~ok]))
print("by ant, vehicle counts per reading:")
print(counts[counts.odor == "vehicle"].pivot_table(index="ant", columns="reading", values="count").to_string())

# Variant: the flagged trials (most of the AL at least 0.2) left out, vehicle trials included
print("\nVariant without the flagged trials (vehicle trials flagged:", int(veh_idx.isin(flagged.index).sum()), ")")
rows = []
for d, v, c in al.READINGS:
    k = d + v + c
    n_veh = 0
    for ant in alarm_ants:
        vi = sel[(sel.ant == ant) & (sel.odor == "vehicle")].index
        vi = vi[~vi.isin(flagged.index)]
        if len(vi) and al.responding(maps, rois[ant, d], vi, v, c)[1] > 0:
            n_veh += 1
    sub = counts_x[counts_x.reading == k]
    rows.append({"reading": k, "ants with a vehicle-responding ROI": n_veh,
                 **{f"ants with >=1 {od} ROI": int((sub[sub.odor == od]["count"] > 0).sum()) for od in al.PHEROMONES}})
fig4b_x = pd.DataFrame(rows)
print(fig4b_x.to_string(index=False))

**Result: not reproduced.** Only biy meets the criterion. Under bix no ant's vehicle trial has a responding ROI (largest vehicle ROI value 0.110), but heptanol and 6MHO each have a responding ROI in only 6 of 13 ants, not most; under biy the vehicle gives none and the four pheromones give 12, 9, 13 and 11 of 13. Under the readings with the ROI maximum (ii) 7 to 10 ants have a vehicle-responding ROI (largest value 1.531), and under aix and aiy one ant does (ant2-m4-d26-y2022, 5 ROIs), whose vehicle trial has 13.5% of the AL at 0.2 or more, against at most 2.4% in the other vehicle trials. No vehicle trial is flagged as a non-specific change (more than half the AL), so the variant without the flagged trial gives the same vehicle counts.

## figs5a-sparse-median-6

> Each alarm pheromone activates few of the roughly 500 glomeruli; even at the highest concentration
> the median number of responding glomeruli is at most 6.

Criterion: counts per ant from M2 (each reading); concentration read as (a) 48% (8 ants) or (b) each
ant's higher concentration (13 ants). Reproduced when, under a reading, the median over ants is at
most 6.6 for every pheromone.

In [ ]:
def medians(cnt):
    out = []
    for k in [d + v + c for d, v, c in al.READINGS]:
        for conc_reading, ants in (("48%", [a for a in alarm_ants if conc_pairs[a][-1] == 0.48]), ("higher", alarm_ants)):
            sub = cnt[(cnt.reading == k) & cnt.ant.isin(ants) & cnt.odor.isin(al.PHEROMONES)]
            med = sub.groupby("odor")["count"].median()
            out.append({"M2 reading": k, "concentration": conc_reading, "n ants": len(ants),
                        **{od: med[od] for od in al.PHEROMONES}, "largest median": med.max(),
                        "<= 6.6": bool(med.max() <= 6.6)})
    return pd.DataFrame(out)
s5a = medians(counts)
print(s5a.to_string(index=False))
print("readings with every median at most 6.6:", int(s5a["<= 6.6"].sum()), "of", len(s5a))
s5a_x = medians(counts_x)
print("\nVariant without the", len(flagged), "trials where most of the AL is at least 0.2:")
print(s5a_x[["M2 reading", "concentration", "largest median", "<= 6.6"]].to_string(index=False))
print("\nPer ant, reading bix (b, mean value, Methods counting):")
print(counts[(counts.reading == "bix") & counts.odor.isin(al.PHEROMONES)].pivot_table(index=["ant", "higher"], columns="odor", values="count").to_string())

**Result: reproduced, resting on the adopted M2 reading.** Under bix the largest median is 3 (hexanol) at 48% (8 ants) and at each ant's higher concentration (13 ants). The four readings b and aix keep every median at or below 6.6 (largest 3 to 6); aiy (7.8 to 8.0) and aiix and aiiy (13 to 14.3) do not. The readings disagree, so the outcome rests on the adopted reading, the ROI's mean value (ImageJ's default measure) with the Methods' trial-mean rule (x), under which both drawings meet the criterion. Leaving out the one flagged trial changes nothing.

## figs5b-shared-glomerulus

> The three compounds that elicit panic alarm activate partly overlapping sets of glomeruli, with a
> single glomerulus, PGb, activated by all three.

Criterion: per ant, the number of M2 ROIs responding to all three (heptanone, heptanol, hexanol) at
(a) each ant's higher concentration or (b) 48%. Reproduced when, under each M2 reading and under (a)
or (b), the median over ants is 1 and more than half of the ants have exactly one such ROI. A
response of one ROI to two odors counts as one glomerulus only at the same depth (Methods).

In [ ]:
def shared_counts(exclude=None):
    rows = []
    for ant in alarm_ants:
        for d, v, c in al.READINGS:
            if exclude is not None:
                g = sel[(sel.ant == ant) & ~sel.index.isin(exclude)]
                R = {od: al.responding(maps, rois[ant, d], g[g.odor == od].index, v, c)[0] for od in al.PHEROMONES}
            else:
                R = {od: resp[ant, d, v, c, od] for od in al.PHEROMONES}
            D = {od: depth[ant, d, od] for od in al.PHEROMONES}
            three = R["heptanone"] & R["heptanol"] & R["hexanol"] & al.same_depth(D["heptanone"], D["heptanol"]) \
                & al.same_depth(D["heptanone"], D["hexanol"]) & al.same_depth(D["heptanol"], D["hexanol"])
            mho = R["6MHO"] & np.logical_or.reduce([R[o] & al.same_depth(D["6MHO"], D[o]) for o in ("heptanone", "heptanol", "hexanol")])
            mho_hk = R["6MHO"] & np.logical_or.reduce([R[o] & al.same_depth(D["6MHO"], D[o]) for o in ("heptanone", "hexanol")])
            rows.append({"ant": ant, "reading": d + v + c, "shared3": int(three.sum()),
                         "pairs": {f"{a}+{b}": int((R[a] & R[b]).sum()) for a, b in (("heptanone", "heptanol"), ("heptanone", "hexanol"), ("heptanol", "hexanol"))},
                         "mho_any": int(mho.sum()), "mho_heptanone_or_hexanol": int(mho_hk.sum())})
    out = pd.DataFrame(rows)
    out["higher"] = out.ant.map(conc_pairs.map(max))
    return out
sh = shared_counts()

def judge_shared(sh):
    rows = []
    for k in sh.reading.unique():
        for conc_reading in ("higher", "48%"):
            sub = sh[(sh.reading == k) & ((sh.higher == 0.48) if conc_reading == "48%" else True)]
            rows.append({"M2 reading": k, "concentration": conc_reading, "n ants": len(sub),
                         "median": sub.shared3.median(), "ants with exactly 1": int((sub.shared3 == 1).sum()),
                         "ants with 0": int((sub.shared3 == 0).sum()), "ants with >1": int((sub.shared3 > 1).sum()),
                         "met": bool(sub.shared3.median() == 1 and (sub.shared3 == 1).sum() > len(sub) / 2)})
    return pd.DataFrame(rows)
s5b = judge_shared(sh)
print(s5b.to_string(index=False))
print("\nPer ant, ROIs responding to all three:")
print(sh.pivot_table(index=["ant", "higher"], columns="reading", values="shared3").to_string())
print("\nPairwise overlaps (ROIs responding to both), reading bix:")
print(pd.DataFrame(list(sh[sh.reading == "bix"].pairs), index=sh[sh.reading == "bix"].ant).to_string())
sh_x = shared_counts(exclude=flagged.index)
print("\nVariant without the flagged trials:")
print(judge_shared(sh_x)[["M2 reading", "concentration", "median", "ants with exactly 1", "met"]].to_string(index=False))

**Result: not reproduced.** Only biy at 48% meets the criterion (median 1, exactly one shared ROI in 5 of 8 ants). Under bix the median is 0 at the higher concentration (exactly one in 4 of 13 ants, none in 8) and 1 at 48% (exactly one in 4 of 8, not more than half). aix and aiy give a median of 1 with exactly one in 5 or 6 of 13 ants, and the readings ii medians of 2 to 2.5. Under bix heptanone and hexanol share an ROI in 9 of 13 ants, so the overlap is partial and the three-way overlap is often missing. Leaving out the flagged trial gives the same.

## figs5c-mho-rarely-overlaps

> 6MHO only rarely and inconsistently activates glomeruli shared with 4-methyl-3-heptanone or
> 4-methyl-3-hexanol.

Criterion: per ant, at (a) its higher concentration or (b) 48%, the number of M2 ROIs responding to
6MHO and to any other pheromone (overlaps at different depths excluded). Reproduced when, under each
M2 reading and under (a) or (b), the median over ants is 0 and the count is non-zero in fewer than
half of the ants.

In [ ]:
def judge_mho(sh, col):
    rows = []
    for k in sh.reading.unique():
        for conc_reading in ("higher", "48%"):
            sub = sh[(sh.reading == k) & ((sh.higher == 0.48) if conc_reading == "48%" else True)]
            rows.append({"M2 reading": k, "concentration": conc_reading, "n ants": len(sub), "median": sub[col].median(),
                         "ants with >0": int((sub[col] > 0).sum()),
                         "met": bool(sub[col].median() == 0 and (sub[col] > 0).sum() < len(sub) / 2)})
    return pd.DataFrame(rows)
s5c = judge_mho(sh, "mho_any")
print("6MHO and any other pheromone (the criterion):")
print(s5c.to_string(index=False))
print("\n6MHO and heptanone or hexanol (the statement's wording):")
print(judge_mho(sh, "mho_heptanone_or_hexanol").to_string(index=False))
print("\nPer ant, ROIs shared by 6MHO and another pheromone:")
print(sh.pivot_table(index=["ant", "higher"], columns="reading", values="mho_any").to_string())
print("\nVariant without the flagged trials:")
print(judge_mho(sh_x, "mho_any")[["M2 reading", "concentration", "median", "ants with >0", "met"]].to_string(index=False))

**Result: reproduced, resting on the adopted M2 reading.** Under bix the median number of ROIs shared by 6MHO and another pheromone is 0, with any in 1 of 13 ants (1 of 8 at 48%). The four readings with the ROI's mean value (aix, aiy, bix, biy) all meet the criterion (shared ROIs in 0 to 2 ants); the four with its maximum (ii) do not (medians 1 to 3, shared ROIs in 11 of 13 ants at the higher concentration). The readings disagree, and the outcome rests on the ROI mean, as in figs5a-sparse-median-6. With heptanone or hexanol only (the statement's wording) the result is the same.

## fig6-shared-panic-pathway (imaging half)

> The three pheromones with overlapping AL responses all elicit panic alarm, while 6MHO does not
> elicit panic and gives a non-overlapping response.

Criterion: reproduced in its imaging half when figs5b-shared-glomerulus and
figs5c-mho-rarely-overlaps are reproduced; the behaviour half rests on fig1e-categorical-response,
whose data are not in the dataset.

In [ ]:
both = s5b.merge(s5c, on=["M2 reading", "concentration"], suffixes=(" shared3", " mho"))
print(both[["M2 reading", "concentration", "met shared3", "met mho"]].to_string(index=False))
print("figs5b met under every M2 reading at the higher concentration:", bool(s5b[s5b.concentration == "higher"].met.all()),
      "; at 48%:", bool(s5b[s5b.concentration == "48%"].met.all()))
print("figs5c met under every M2 reading at the higher concentration:", bool(s5c[s5c.concentration == "higher"].met.all()),
      "; at 48%:", bool(s5c[s5c.concentration == "48%"].met.all()))

**Result: not reproduced, imaging half only.** figs5b-shared-glomerulus is met under no reading at the higher concentration and only under biy at 48%; figs5c-mho-rarely-overlaps is met under the four mean-value readings. The behaviour half rests on fig1e-categorical-response, whose data are not in the dataset.

## fig4-no-decreases

> Odors cause no fluorescence decreases; only small non-specific decreases from shifts in AL position
> and photobleaching occur.

Criterion: per-plane peak fold change maps of the first look (stabilization, 1-sigma Gaussian blur);
reproduced when no glomerulus-sized region (10 to 20 um) reaches -0.2 or below in at least two trials of
the same odor and concentration without doing so in that ant's vehicle trial. Only each ant's higher
concentration is mapped here. A region is counted when it is at least 10 um across (the lower end of a
glomerulus); larger regions are counted too, and their sizes are shown. "Doing so in the vehicle
trial" is read as at least half of the region at -0.2 or below in the vehicle map of the same plane.

In [ ]:
min_bins = np.pi * (al.GLOM_MIN_UM / 2) ** 2 / (al.PIXEL_UM * al.BIN) ** 2
dec = []
for ant in alarm_ants:
    g = sel[sel.ant == ant]
    f0mean = np.mean([maps[i]["f0"] for i in g.index], axis=0)
    from skimage.filters import threshold_otsu
    thr = threshold_otsu(np.log1p(f0mean.max(0)))
    vi = g[g.odor == "vehicle"].index[0]
    for od in al.PHEROMONES + ["isopropanol"]:
        idx = g[g.odor == od].index
        if len(idx) < 2:
            continue
        blur = np.stack([maps[i]["blur"] for i in idx])
        for p in range(al.N_PLANES):
            plane_mask = np.log1p(f0mean[p]) > thr
            edge_um = ndimage.distance_transform_edt(plane_mask) * al.PIXEL_UM * al.BIN
            neg = ((blur[:, p] <= -al.THRESHOLD).sum(0) >= 2) & plane_mask
            lab, n = ndimage.label(neg)
            for k in range(1, n + 1):
                reg = lab == k
                if reg.sum() < min_bins:
                    continue
                in_vehicle = (maps[vi]["blur"][p][reg] <= -al.THRESHOLD).mean() >= 0.5
                dec.append({"ant": ant, "odor": od, "plane": p, "diameter_um": 2 * np.sqrt(reg.sum() / np.pi) * al.PIXEL_UM * al.BIN,
                            "mean_value": float(blur[:, p][:, reg].mean()), "in_vehicle": bool(in_vehicle),
                            "edge_um": float(edge_um[reg].min()), "mask": reg})
dec = pd.DataFrame(dec, columns=["ant", "odor", "plane", "diameter_um", "mean_value", "in_vehicle", "edge_um", "mask"])
odor_dec = dec[~dec.in_vehicle]
print("glomerulus-sized regions at -0.2 or below in >=2 trials of one odor:", len(dec),
      "; of these also in the vehicle trial:", int(dec.in_vehicle.sum()), "; odor-specific:", len(odor_dec))
print("odor-specific regions per ant and odor:")
print(odor_dec.groupby(["ant", "odor"]).size().unstack(fill_value=0).to_string() if len(odor_dec) else "none")
if len(odor_dec):
    print("diameters (um): median", odor_dec.diameter_um.median().round(1), "range", odor_dec.diameter_um.min().round(1), "to", odor_dec.diameter_um.max().round(1))
    print("regions of 10 to 20 um:", int(odor_dec.diameter_um.between(10, 20).sum()))
    print("most negative odor-specific region:")
    print(odor_dec.drop(columns="mask").sort_values("mean_value").head(5).round(3).to_string(index=False))
    print("ants with at least one:", odor_dec.ant.nunique(), "of", len(alarm_ants))

# Where the odor-specific negative regions lie: on the AL border (motion moves the edge of the
# fluorescent lobe), and in the same place for other odors (a decrease specific to an odor would not be)
if len(odor_dec):
    other = []
    for k, r in odor_dec.iterrows():
        same = odor_dec[(odor_dec.ant == r.ant) & (odor_dec.plane == r.plane) & (odor_dec.odor != r.odor)]
        u = np.logical_or.reduce(list(same["mask"])) if len(same) else np.zeros_like(r["mask"])
        other.append((u & r["mask"]).sum() / r["mask"].sum())
    odor_dec = odor_dec.assign(overlap_other_odor=other)
    print("odor-specific regions overlapping a negative region of another odor (same ant and plane):",
          int((odor_dec.overlap_other_odor > 0).sum()), "of", len(odor_dec), "; by half or more:", int((odor_dec.overlap_other_odor >= 0.5).sum()))
    print("distance of the region's nearest bin to the AL border (um): median", round(odor_dec.edge_um.median(), 1),
          "; regions touching the border (within one bin, 1.2 um):", int((odor_dec.edge_um <= al.PIXEL_UM * al.BIN).sum()),
          "; within 5 um:", int((odor_dec.edge_um <= 5).sum()), "of", len(odor_dec))
    # reference: how often a 10 um region centred at random in a plane's AL mask comes within 5 um of the border
    ref = []
    for ant in alarm_ants:
        g = sel[sel.ant == ant]
        f0mean = np.mean([maps[i]["f0"] for i in g.index], axis=0)
        thr = threshold_otsu(np.log1p(f0mean.max(0)))
        for p in range(al.N_PLANES):
            pm = np.log1p(f0mean[p]) > thr
            if pm.sum() > 100:
                ref.append((ndimage.distance_transform_edt(pm)[pm] * al.PIXEL_UM * al.BIN <= 5 + al.GLOM_MIN_UM / 2).mean())
    print("reference: fraction of AL bins within 10 um of the border (a 10 um region centred there comes within 5 um), mean over planes:", round(float(np.mean(ref)), 3))
    print("by odor: regions, overlapping another odor's, touching the border")
    print(odor_dec.groupby("odor").agg(n=("plane", "size"), overlap=("overlap_other_odor", lambda s: int((s > 0).sum())),
                                       border=("edge_um", lambda s: int((s <= al.PIXEL_UM * al.BIN).sum()))).to_string())

**Result: not reproduced.** 594 regions at least 10 um across (540 of them 10 to 20 um) reach -0.2 or below in at least two trials of one odor without doing so in the vehicle trial, in all 13 ants; the most negative has a mean of -0.452 (ant4-m4-d26-y2022, hexanol, plane 3). Where they lie points to motion, the source the paper names for small non-specific decreases: 487 of the 594 touch the AL border (within one 1.2 um bin) and 564 lie within 5 um of it, against 83% of the AL area expected for a 10 um region placed at random; and 384 overlap a negative region of another odor in the same plane (302 by half or more). The criterion counts every such region, and its vehicle comparison rests on one vehicle trial per ant.

## fig4-large-lasting-responses

> Fluorescence increases to alarm pheromones are often large (1 to 2-fold change) and outlast the 5 s
> odor pulse.

Criterion: over responding ROI-trials (M2) at each ant's higher concentration, the median ROI peak
fold change lies in 0.9 to 2.2, under each ROI value reading (i) and (ii); and in more than half of
those ROI-trials dF/F0 at the first volume after the pulse ends (8.4 s, volume 8) is still above the
baseline mean plus 3 SD. A responding ROI-trial is an ROI region whose value in that trial is at least
0.2 (counting y). The trace is the ROI's mean raw fluorescence in the plane of its depth.

In [ ]:
rows = []
for ant in alarm_ants:
    g = sel[(sel.ant == ant) & sel.odor.isin(al.PHEROMONES)]
    for d in "ab":
        for r_ in rois[ant, d]:
            for i in g.index:
                vi_, vii_ = al.roi_values(maps, [r_], i)
                fr = maps[i]["frames"][r_["depth"]].astype(np.float32)   # volumes 1-5 and 8
                f = fr[:, r_["mask"]].mean(1)
                dff = f / f[:5].mean() - 1
                lasting = dff[5] > dff[:5].mean() + 3 * dff[:5].std(ddof=1)
                rows.append({"ant": ant, "drawing": d, "odor": sel.loc[i].odor, "vi": vi_[0], "vii": vii_[0], "lasting": lasting})
lr = pd.DataFrame(rows)
out = []
for d in "ab":
    for v in ("vi", "vii"):
        sub = lr[(lr.drawing == d) & (lr[v] >= al.THRESHOLD)]
        out.append({"drawing": d, "value": v[1:], "responding ROI-trials": len(sub), "median peak": round(sub[v].median(), 3),
                    "IQR": (round(sub[v].quantile(0.25), 3), round(sub[v].quantile(0.75), 3)),
                    "fraction in 1-2": round(sub[v].between(1, 2).mean(), 3),
                    "fraction lasting past 8 s": round(sub.lasting.mean(), 3), "ants": sub.ant.nunique(),
                    "median in 0.9-2.2": bool(0.9 <= sub[v].median() <= 2.2), "lasting > half": bool(sub.lasting.mean() > 0.5)})
lrt = pd.DataFrame(out)
print(lrt.to_string(index=False))
print("\nBy ant (drawing b, value i):")
sub = lr[(lr.drawing == "b") & (lr.vi >= al.THRESHOLD)]
print(sub.groupby("ant").agg(n=("vi", "size"), median=("vi", "median"), lasting=("lasting", "mean")).round(3).to_string())

**Result: not reproduced.** No reading meets both parts. With glomerulus-sized ROIs (b) the median peak fold change of responding ROI-trials is 0.453 with the mean value (445 ROI-trials, 70.3% still elevated at 8.4 s) and 1.016 with the maximum (1,086 ROI-trials, 41.3% elevated): the first median is below 0.9 and the second reading fails the lasting part. With drawing a the medians are 0.377 and 0.573. Under the four readings 6.3 to 32.7% of responding ROI-trials lie within 1 to 2.

## fig5ab-focal-glomeruli

> PGb lies in the anterior ventral hemi-lobe next to a gap without glomeruli, about halfway between the
> dorsal and ventral AL surfaces, with two neighbouring alarm-responsive glomeruli in the same plane;
> PGb and 6G were found in 13 of 13 ants and PGa in 11 of 13.

Criterion: identification by M4 on each ant's higher-concentration ROIs (M2, each reading). Reproduced
when the counts are within tolerance (12 or 13, 12 or 13, 10 to 12) and, in most ants with PGb, its
centre plane lies in the middle third of the AL depth (from the top plane to the deepest plane with AL
fluorescence). "Adjacent" (M4) is read as centres at most 20 um apart; the deepest plane with AL
fluorescence as the deepest where the mean baseline fluorescence inside the AL mask is at least 25% of
the brightest plane's (10% and 50% shown).

In [ ]:
focal, idrows = {}, []
for ant in alarm_ants:
    hi = sel[(sel.ant == ant) & sel.odor.isin(al.PHEROMONES)]
    f0prof = np.mean([maps[i]["f0"][:, al_masks[ant]].mean(1) for i in sel[sel.ant == ant].index], axis=0)
    deepest = {q: int(np.max(np.where(f0prof >= q * f0prof.max())[0])) for q in (0.1, 0.25, 0.5)}
    for d, v, c in al.READINGS:
        f = al.identify_focal(maps, rois[ant, d], hi, v, c)
        focal[ant, d + v + c] = f
        row = {"ant": ant, "reading": d + v + c, "PGb": f["PGb"] is not None, "6G": f["6G"] is not None,
               "PGa": f["PGa"] is not None, "PGb candidates": f["n_pgb_candidates"]}
        if f["PGb"] is not None:
            p = f["depth"]["PGb"]
            row.update({"PGb plane": p, "AL deepest plane": deepest[0.25], "PGb depth fraction": round(p / deepest[0.25], 2)})
            for q in (0.1, 0.25, 0.5):
                row[f"middle third ({int(q * 100)}%)"] = deepest[q] / 3 <= p <= 2 * deepest[q] / 3
            row["6G plane"] = f["depth"].get("6G")
            row["PGa plane"] = f["depth"].get("PGa")
        idrows.append(row)
ident = pd.DataFrame(idrows)
summ = ident.groupby("reading").agg(PGb=("PGb", "sum"), G6=("6G", "sum"), PGa=("PGa", "sum"),
                                    middle_third=("middle third (25%)", "sum"),
                                    middle_third_10=("middle third (10%)", "sum"), middle_third_50=("middle third (50%)", "sum"))
summ["counts within tolerance"] = (summ.PGb >= 12) & (summ.G6 >= 12) & summ.PGa.between(10, 12)
summ["PGb in middle third in most"] = summ.middle_third > summ.PGb / 2
print(summ.to_string())
print("\nPer ant, reading bix:")
print(ident[ident.reading == "bix"].drop(columns=["reading"]).to_string(index=False))
MAIN = "bix"
# A second identification for the traces: the reading that identifies PGa in the most ants (then PGb and 6G)
ALT = (summ.PGa * 1000 + summ.PGb * 10 + summ.G6).idxmax()
print("\nsingle-glomerulus traces are computed under", MAIN, "(the adopted M2 reading) and", ALT,
      "(the reading that identifies PGa in the most ants):", summ.loc[ALT, ["PGb", "G6", "PGa"]].to_dict())

**Result: not reproduced.** No reading gives all three counts within tolerance: PGb is identified in 5 to 12 of 13 ants, 6G in 1 to 10 and PGa in 0 to 4 (bix 5, 2, 0; aiix 12, 10, 3). With the deepest plane at 25% of the brightest plane's AL fluorescence, PGb lies in the middle third of the depth in 1 to 3 ants per reading (bix 1 of 5, the others at 0.71 to 0.77 of the depth); with a 10% rule, in 5 to 11.

## fig5d-stereotyped-positions

> PGa is always anterior (12.9 +/- 1.9 SD um) and slightly lateral (5.1 +/- 2.9 um) to PGb, and 6G always
> lateral to PGb (13.1 +/- 2.6 um) at a similar anterior-posterior position (0.6 +/- 2.2 um).

Criterion: one plane per ant (PGb's), PGb at (0, 0); centres (a) the centroid of each glomerulus's
pixels of at least 0.2 to its defining odor in that plane, inside its ROI, or (b) its peak pixel; the
plane's map from (c) one higher-concentration trial (the first) or (d) the mean over that ant's
higher-concentration trials. Reproduced when, under every reading, the four means are within
tolerance, PGa is anterior and 6G lateral in every ant, and each SD is below 10 um. Glomeruli are
identified by M4 under the main M2 reading (b, i, x: glomerulus-sized ROIs, ImageJ's mean, the
Methods' counting); the other M2 readings are shown as variants. Note that M4 requires PGa anterior
and 6G lateral to PGb, so the "every ant" conditions hold by construction for identified glomeruli.

In [ ]:
DEFINING = {"PGb": ["heptanone", "heptanol", "hexanol"], "6G": ["6MHO"], "PGa": ["heptanol", "hexanol"]}

def centres(ant, f, rd_centre, rd_plane, d):
    hi = sel[(sel.ant == ant) & sel.odor.isin(al.PHEROMONES)]
    p = f["depth"]["PGb"]
    masks = al.group_masks(rois[ant, d])
    out = {}
    for name, odors in DEFINING.items():
        if f[name] is None:
            continue
        idx = [hi[hi.odor == o].index for o in odors]
        if rd_plane == "c":
            m = np.mean([maps[ix[0]]["planes"][p] for ix in idx], axis=0)
        else:
            m = np.mean([maps[i]["planes"][p] for ix in idx for i in ix], axis=0)
        roi = ndimage.binary_dilation(masks[f[name]], iterations=2)
        if rd_centre == "a":
            px = roi & (m >= al.THRESHOLD)
            if not px.any():
                px = roi
            out[name] = al.to_full(ndimage.center_of_mass(np.where(px, m, 0)))
        else:
            out[name] = al.to_full(np.unravel_index(np.argmax(np.where(roi, m, -np.inf)), m.shape))
    return out

pos = []
for reading_m2 in [d + v + c for d, v, c in al.READINGS]:
    for ant in alarm_ants:
        f = focal[ant, reading_m2]
        if f["PGb"] is None:
            continue
        for rc in "ab":
            for rp in "cd":
                cs = centres(ant, f, rc, rp, reading_m2[0])
                for name in ("PGa", "6G"):
                    if name in cs:
                        a, l = al.offsets_um(cs["PGb"], cs[name])
                        pos.append({"M2": reading_m2, "centre": rc, "plane": rp, "ant": ant, "glom": name, "anterior": a, "lateral": l})
pos = pd.DataFrame(pos)
EXPECTED = {("PGa", "anterior"): (12.9, 1.9), ("PGa", "lateral"): (5.1, 2.9), ("6G", "lateral"): (13.1, 2.6), ("6G", "anterior"): (0.6, 2.2)}

def judge_pos(p):
    rows = []
    for (m2, rc, rp), g in p.groupby(["M2", "centre", "plane"]):
        row = {"M2": m2, "reading": rc + rp}
        ok = True
        for (glom, axis), (val, tol) in EXPECTED.items():
            s = g[g.glom == glom][axis]
            row[f"{glom} {axis}"] = f"{s.mean():.1f} +/- {s.std():.1f} (n={len(s)})"
            ok &= bool(abs(s.mean() - val) <= tol and s.std() < 10) if len(s) > 1 else False
        row["PGa anterior in all"] = bool((g[g.glom == "PGa"].anterior > 0).all())
        row["6G lateral in all"] = bool((g[g.glom == "6G"].lateral > 0).all())
        row["met"] = ok and row["PGa anterior in all"] and row["6G lateral in all"]
        rows.append(row)
    return pd.DataFrame(rows)
jp = judge_pos(pos)
print("Main M2 reading (bix):")
print(jp[jp.M2 == MAIN].to_string(index=False))
print("\nPer ant, bix, centre a, plane d (um):")
print(pos[(pos.M2 == MAIN) & (pos.centre == "a") & (pos.plane == "d")].pivot_table(index="ant", columns="glom", values=["anterior", "lateral"]).round(1).to_string())
print("\nOther M2 readings:")
print(jp[jp.M2 != MAIN].to_string(index=False))

# Orientation check: baseline fluorescence in PGb's plane for one ant where all three are identified
# under ALT, with the centres marked (rows: up is lateral; columns: right is anterior, as the files say)
ant_o = next((a for a in alarm_ants if all(focal[a, ALT][n] is not None for n in ("PGb", "6G", "PGa"))), None)
if ant_o is not None:
    f_o = focal[ant_o, ALT]
    p_o = f_o["depth"]["PGb"]
    img = np.mean([maps[i]["f0"][p_o] for i in sel[sel.ant == ant_o].index], axis=0)
    fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
    ax[0].imshow(img, cmap="gray")
    hi = sel[(sel.ant == ant_o) & sel.odor.isin(al.PHEROMONES)]
    ax[1].imshow(np.mean([maps[i]["planes"][p_o] for i in hi[hi.odor.isin(["heptanol", "hexanol"])].index], axis=0), cmap="magma", vmin=0, vmax=1)
    for a_ in ax:
        for n, col in (("PGb", "c"), ("6G", "lime"), ("PGa", "y")):
            r_, c_ = f_o["centroid"][n]
            a_.plot(c_ / al.BIN, r_ / al.BIN, "o", mfc="none", mec=col, ms=8)
            a_.text(c_ / al.BIN + 4, r_ / al.BIN, n, color=col, fontsize=8)
        a_.set_xlabel("column (anterior to the right)"); a_.set_ylabel("row (lateral up)")
    ax[0].set_title(f"{ant_o}, plane {p_o}: baseline F0"); ax[1].set_title("heptanol and hexanol peak map (0 to 1)")
    plt.tight_layout(); plt.show()
    print(ant_o, "plane", p_o, "centres (row, column, full-resolution pixels):",
          {n: tuple(round(float(x), 1) for x in f_o["centroid"][n]) for n in ("PGb", "6G", "PGa")})
    print("binned F0 image range", round(float(img.min()), 1), "to", round(float(img.max()), 1))

**Result: not reproduced.** No reading meets the criterion, so the readings do not disagree on the outcome. Under bix no PGa is identified, and 6G (2 ants) lies 12.9 to 15.6 um lateral and 0.1 to 3.5 um anterior of PGb on average. Under the other readings PGa (2 to 4 ants) lies 9.2 to 18.5 um anterior of PGb on average (expected 12.9 +/- 1.9) with SDs up to 13.1 um, and 0.4 to 9.8 um lateral with SDs up to 14.0 um; 6G lies 7.6 to 20.8 um lateral (1 to 10 ants) with SDs of 1.3 to 8.8 um. In the baseline image of ant1-m4-d26-y2022, PGa lies to the right of PGb (anterior) and 6G above it (lateral), the arrangement of the paper's Figure 5B under the files' orientation.

## Single-glomerulus traces (M3)

For each ant where PGb was identified under the main M2 reading (bix) or under the reading that
identifies PGa in the most ants (ALT, printed above), every pheromone trial at both concentrations,
the vehicle trial and the 144 s recordings are read as row bands around the focal glomeruli, on PGb's,
6G's and PGa's planes +-2. Each plane's band is stabilized as the maps are (phase correlation against
the baseline mean), as the claims file's M3 assumes; the unstabilized disc trace is kept to show what
this changes. The centre plane of each glomerulus is chosen per trial among its higher-concentration
depth +-1, (a) where its area of at least 0.2 is largest or (b) where its disc's peak fold change is
highest. The ROI is the claims file's 15 um disc on the centre ("disc") or the region of at least 0.2
there ("region"). Every claim below is shown under both identifications, both plane readings and
both ROIs. Volume times are (index - 1) x 1.205 s plus the centre plane's offset within the volume
scan (`PLANE_DT` x plane, from the files' `starting_time`).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
m3_trials = alarm[alarm.odor.isin(al.PHEROMONES + ["vehicle"])]
IDS = [MAIN] + ([ALT] if ALT != MAIN else [])
jobs = []
for ant in alarm_ants:
    fs = {k: focal[ant, k] for k in IDS if focal[ant, k]["PGb"] is not None}
    if not fs:
        continue
    rows_band = al.band_for([c for f in fs.values() for c in f["centroid"].values()])
    for idx, r in m3_trials[m3_trials.ant == ant].iterrows():
        jobs.append((idx, r, rows_band, fs))
print(len(jobs), "trials for M3, from", len({j[1].ant for j in jobs}), "ants")
t = time.time()

def run(job):
    idx, r, rows_band, fs = job
    return idx, al.trial_m3(r.blobs, r.nvol, rows_band, fs)
with ThreadPoolExecutor(6) as ex:
    m3 = dict(ex.map(run, jobs))
print(f"M3 in {time.time() - t:.0f} s")
trace_rows, shift_rows = [], []
for idx, res in m3.items():
    r = m3_trials.loc[idx]
    for p, s_ in res["_shift"].items():
        shift_rows.append({"ant": r.ant, "trial": r.trial, "odor": r.odor, "nvol": r.nvol, "plane": p,
                           "max shift px": int(np.abs(s_).max()), "shifts": s_})
    for ident in IDS:
        for name, rr in res.get(ident, {}).items():
            for (rd, roi), v in rr.items():
                trace_rows.append({"ident": ident, "ant": r.ant, "trial": r.trial, "odor": r.odor, "conc": r.conc, "nvol": r.nvol,
                                   "glom": name, "plane_reading": rd, "roi": roi, "plane": v["plane"], "peak": v["peak"],
                                   "roi_px": v["roi_px"], "t0": v["plane"] * PLANE_DT, "trace": v["trace"]})
tr_all = pd.DataFrame(trace_rows)
shifts_m3 = pd.DataFrame(shift_rows)
tr = tr_all[tr_all.roi != "raw"]
print(tr.groupby(["ident", "glom", "plane_reading", "roi"]).size().unstack(["plane_reading", "roi"]).to_string())
print("ants with traces:", tr.groupby(["ident", "glom"]).ant.nunique().unstack().to_string())
print("region ROI size (pixels; the disc has", int(tr[tr.roi == "disc"].roi_px.iloc[0]), "): median",
      int(tr[tr.roi == "region"].roi_px.median()), "; trials where no disc pixel reached 0.2, so the disc was used:",
      int((tr[tr.roi == "region"].roi_px.values == tr[tr.roi == "disc"].roi_px.values).sum()), "of", int((tr.roi == "region").sum()))

In [ ]:
# Clock check: the odor-evoked rise follows the stimulus at 3 s in pheromone trials and is absent in the
# vehicle trials (PGb, MAIN identification, plane reading b, disc, 40-volume trials at the higher concentration)
t40 = np.arange(40) * al.VOLUME_S
sub = tr[(tr.ident == MAIN) & (tr.glom == "PGb") & (tr.plane_reading == "b") & (tr.roi == "disc") & (tr.nvol == 40)]
odor_tr = np.stack(sub[sub.odor.isin(["heptanone", "heptanol", "hexanol"]) & (sub.conc == sub.ant.map(conc_pairs.map(max)))].trace)
veh_tr = np.stack(sub[sub.odor == "vehicle"].trace)
print("volume time (s):      ", np.round(t40[:14], 1).tolist())
print("PGb mean dF/F0, odor: ", np.round(odor_tr.mean(0)[:14].astype(float), 3).tolist(), "n =", len(odor_tr))
print("PGb mean dF/F0, vehicle:", np.round(veh_tr.mean(0)[:14].astype(float), 3).tolist(), "n =", len(veh_tr))
diff = odor_tr.mean(0) - veh_tr.mean(0)
print("first volume where odor minus vehicle exceeds 0.1:", int(np.argmax(diff > 0.1)) + 1, "at", round(t40[np.argmax(diff > 0.1)], 1), "s")
print("odor minus vehicle over volumes 1-3 (before 3 s):", np.round(diff[:3].astype(float), 3).tolist())

# Motion in the row bands: the shift of each volume against the baseline mean, and what stabilization changes
mx = shifts_m3.groupby(["ant", "trial"])["max shift px"].max()
print("\nlargest shift per trial in the M3 bands (pixels; 0.289 um each): median", mx.median(), "max", mx.max(),
      "; trials over 10 pixels:", int((mx > 10).sum()), "of", len(mx))
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ex = shifts_m3[shifts_m3.nvol == 40].sort_values("max shift px")
for k in list(range(0, len(ex), max(1, len(ex) // 3)))[:3] + [len(ex) - 1]:
    s_ = ex.iloc[k]
    ax[0].plot(np.arange(1, 41), np.hypot(*s_.shifts.T) * al.PIXEL_UM, label=f"{s_.ant[:9]} t{s_.trial} {s_.odor} plane {s_.plane}")
ax[0].axvspan(3.5, 6.5, color="0.9"); ax[0].set_xlabel("volume"); ax[0].set_ylabel("shift (um)"); ax[0].legend(fontsize=6)
st = tr_all[(tr_all.ident == MAIN) & (tr_all.glom == "PGb") & (tr_all.plane_reading == "b") & tr_all.roi.isin(["disc", "raw"])]
pv = st.pivot_table(index=["ant", "trial"], columns="roi", values="peak")
print("PGb peak fold change (MAIN, plane b, disc), stabilized against not: median absolute difference",
      round(float((pv.disc - pv.raw).abs().median()), 3), "; largest", round(float((pv.disc - pv.raw).abs().max()), 3),
      "; r =", round(float(np.corrcoef(pv.disc, pv.raw)[0, 1]), 3), "; trials on different sides of 0.2:",
      int(((pv.disc >= 0.2) != (pv.raw >= 0.2)).sum()), "of", len(pv))
k_ = (pv.disc - pv.raw).abs().idxmax()
for roi, sty in (("disc", "k"), ("raw", "r--")):
    ax[1].plot(t40, st[(st.ant == k_[0]) & (st.trial == k_[1]) & (st.roi == roi)].trace.iloc[0][:40], sty, label=f"{roi} ({'stabilized' if roi == 'disc' else 'not stabilized'})")
ax[1].set_title(f"PGb, {k_[0][:9]} trial {k_[1]}: the largest difference", fontsize=8); ax[1].set_xlabel("s"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## fig5c-s6-tuning

> PGb, PGa and 6G each respond to a unique combination of pheromones, with several pheromone and
> glomerulus pairs giving no response; PGb responds to heptanone, heptanol and hexanol and not to
> 6MHO, PGa to heptanol and hexanol, and 6G to 6MHO.

Criterion: peak fold change per trial from M3, all concentrations; readings (a) a linear model with
individual as a fixed factor, (b) a linear mixed model with a random intercept per ant;
concentration log10-transformed. A combination responds when the prediction at 48% is at least 0.2
with its 95% CI above 0, and gives no response when the 95% CI at 48% includes 0. Reproduced when the
six named combinations respond, PGb to 6MHO does not, and the three glomeruli's sets of responding
pheromones all differ.

In [ ]:
x48 = np.log10(0.48)
fits = []
for ident in IDS:
    for roi in ("disc", "region"):
        for rd in "ab":
            for glom in ("PGb", "PGa", "6G"):
                for od in al.PHEROMONES:
                    d_ = tr[(tr.ident == ident) & (tr.roi == roi) & (tr.glom == glom) & (tr.odor == od) & (tr.plane_reading == rd) & (tr.nvol == 40)].copy()
                    d_["lc"] = np.log10(d_.conc)
                    row = {"ident": ident, "roi": roi, "plane": rd, "glom": glom, "odor": od, "n trials": len(d_), "n ants": d_.ant.nunique()}
                    if d_.ant.nunique() < 2:
                        fits.append(row)
                        continue
                    # (a) fixed ant effects, sum coding: intercept + slope*x is the prediction for the average ant
                    fa = smf.ols("peak ~ lc + C(ant, Sum)", d_).fit()
                    xv = np.zeros(len(fa.params)); xv[0] = 1; xv[list(fa.params.index).index("lc")] = x48
                    pa = float(xv @ fa.params); sa = float(np.sqrt(xv @ fa.cov_params().values @ xv))
                    q = stats.t.ppf(0.975, fa.df_resid)
                    row.update({"(a) pred": pa, "(a) lo": pa - q * sa, "(a) hi": pa + q * sa})
                    try:
                        fb = smf.mixedlm("peak ~ lc", d_, groups=d_.ant).fit(reml=True)
                        fe = fb.fe_params; cv = fb.cov_params().loc[fe.index, fe.index].values
                        xb = np.array([1, x48]); pb = float(xb @ fe); sb = float(np.sqrt(xb @ cv @ xb))
                        row.update({"(b) pred": pb, "(b) lo": pb - 1.96 * sb, "(b) hi": pb + 1.96 * sb, "(b) converged": bool(fb.converged)})
                    except Exception as e:  # noqa: BLE001
                        row.update({"(b) pred": np.nan, "(b) lo": np.nan, "(b) hi": np.nan, "error": str(e)[:60]})
                    fits.append(row)
fits = pd.DataFrame(fits)
for m in ("(a)", "(b)"):
    fits[f"{m} responds"] = (fits[f"{m} pred"] >= 0.2) & (fits[f"{m} lo"] > 0)
    fits[f"{m} no response"] = (fits[f"{m} lo"] <= 0) & (fits[f"{m} hi"] >= 0)
print("fits with fewer than 2 ants (not fitted):", int(fits["(a) pred"].isna().sum()), "of", len(fits), "; mixed-model fits that failed:",
      int(fits["error"].notna().sum()) if "error" in fits else 0, "; mixed-model fits not converged:", int((fits["(b) converged"] == False).sum()))
print(fits.drop(columns=[c for c in ["error"] if c in fits]).round(3).to_string(index=False))
NAMED = [("PGb", "heptanone"), ("PGb", "heptanol"), ("PGb", "hexanol"), ("PGa", "heptanol"), ("PGa", "hexanol"), ("6G", "6MHO")]
tun = []
for ident in IDS:
    for roi in ("disc", "region"):
        for rd in "ab":
            for m in ("(a)", "(b)"):
                f_ = fits[(fits.ident == ident) & (fits.roi == roi) & (fits.plane == rd)].set_index(["glom", "odor"]).reindex(
                    pd.MultiIndex.from_product([["PGb", "PGa", "6G"], al.PHEROMONES]))
                f_[[f"{m} responds", f"{m} no response"]] = f_[[f"{m} responds", f"{m} no response"]].fillna(False).astype(bool)
                named = [bool(f_.loc[k, f"{m} responds"]) for k in NAMED]
                pgb_mho = bool(f_.loc[("PGb", "6MHO"), f"{m} no response"])
                sets = {g: frozenset(o for o in al.PHEROMONES if f_.loc[(g, o), f"{m} responds"]) for g in ("PGb", "PGa", "6G")}
                distinct = len(set(sets.values())) == 3
                tun.append({"ident": ident, "roi": roi, "plane": rd, "model": m, "named responding": f"{sum(named)}/6",
                            "which": "".join("1" if x else "0" for x in named), "PGb-6MHO none": pgb_mho,
                            "PGb set": ",".join(sorted(sets["PGb"])), "PGa set": ",".join(sorted(sets["PGa"])),
                            "6G set": ",".join(sorted(sets["6G"])), "sets differ": distinct,
                            "met": all(named) and pgb_mho and distinct})
tun = pd.DataFrame(tun)
print("\n'which' marks the six named combinations in the order", NAMED)
print(tun.to_string(index=False))

**Result: partially reproduced, resting on the identification and the disc ROI.** With the 15 um disc, PGb responds at 48% to heptanone, heptanol and hexanol under both identifications, both plane readings and both models (plane a, model a: bix 0.851, 0.678, 0.734; aiiy 0.596, 0.327, 0.528), and 6G responds to 6MHO (0.335 and 0.280). PGa is not identified under bix. Under aiiy (4 ants) it responds to heptanol (0.258 to 0.274) and its hexanol prediction (0.164 to 0.191, CI above 0) falls short of 0.2, so 5 of the 6 named combinations respond. PGb's prediction for 6MHO is negative with its CI below 0 (-0.087 and -0.084), which the criterion counts as neither a response nor no response. With the region of at least 0.2, which takes in neighbouring glomeruli within 17.5 um of the centre, PGb responds to 6MHO in some fits and under aiiy the three sets of responding pheromones differ in only one of four fits. No reading meets the whole criterion.

## fig5c-6g-slow

> Responses in PGb and PGa peak and then decline close to baseline by the end of the 48 s recording,
> while 6G responses to 6MHO are very slow, with a plateau of tens of seconds.

Criterion: mean M3 traces for heptanone in PGb, heptanol and hexanol in PGb and PGa, 6MHO in 6G;
readings (a) 48% (8 ants) or (b) all concentrations pooled. Reproduced when the PGb and PGa mean
traces over the last five volumes are below 0.2 and below a quarter of their peak, while the 6G trace
stays at or above half its peak for at least 20 s.

In [ ]:
COMBOS = [("PGb", "heptanone"), ("PGb", "heptanol"), ("PGb", "hexanol"), ("PGa", "heptanol"), ("PGa", "hexanol"), ("6G", "6MHO")]
rows = []
for ident in IDS:
    for roi in ("disc", "region"):
        for rd in "ab":
            for conc_reading in ("48%", "pooled"):
                for glom, od in COMBOS:
                    s = tr[(tr.ident == ident) & (tr.roi == roi) & (tr.glom == glom) & (tr.odor == od) & (tr.plane_reading == rd) & (tr.nvol == 40)]
                    if conc_reading == "48%":
                        s = s[s.conc == 0.48]
                    if not len(s):
                        continue
                    m = np.stack(s.trace).mean(0)
                    pk = m.max()
                    end = m[-5:].mean()
                    above = m >= pk / 2
                    # longest run of volumes at or above half the peak
                    run_, best = 0, 0
                    for a in above:
                        run_ = run_ + 1 if a else 0
                        best = max(best, run_)
                    rows.append({"ident": ident, "roi": roi, "plane": rd, "conc": conc_reading, "glom": glom, "odor": od, "n": len(s),
                                 "ants": s.ant.nunique(), "peak": round(pk, 3), "peak volume": int(m.argmax()) + 1, "last 5 mean": round(end, 3),
                                 "end/peak": round(end / pk, 3), "s at >= half peak": round(best * al.VOLUME_S, 1)})
slow = pd.DataFrame(rows)
print(slow.to_string(index=False))
slow_j = []
for (ident, roi, rd, cr), g in slow.groupby(["ident", "roi", "plane", "conc"]):
    fast = g[g.glom != "6G"]
    six = g[g.glom == "6G"]
    ok_fast = bool(((fast["last 5 mean"] < 0.2) & (fast["end/peak"] < 0.25)).all())
    ok_six = bool((six["s at >= half peak"] >= 20).all()) if len(six) else False
    slow_j.append({"ident": ident, "roi": roi, "plane": rd, "conc": cr, "PGa tested": bool((fast.glom == "PGa").any()),
                   "PGb/PGa near baseline": ok_fast, "6G >= half peak for >= 20 s": ok_six, "met": ok_fast and ok_six})
slow_j = pd.DataFrame(slow_j)
print(slow_j.to_string(index=False))

**Result: partially reproduced, resting on the identification and the disc ROI.** With the disc, the mean PGb traces for heptanone, heptanol and hexanol end (last five volumes) at -0.165 to 0.116, with end over peak at most 0.151, and the mean 6G trace to 6MHO stays at or above half its peak for 36.1 to 37.3 s, under both identifications, both plane readings and both concentration readings. PGa is identified only under aiiy, and its traces (9 trials at 48%, 24 pooled) end at -0.149 to 0.046; its hexanol trace peaks at only 0.129 to 0.148. With the region ROI the PGb heptanol trace ends at 0.272 to 0.472, above 0.2, and the criterion fails. The PGa part rests on 3 or 4 ants under one identification.

## figs7a-6g-extended

> In 144 s recordings, 6G responses to 6MHO plateau for about 30 s and return to baseline about 80 s
> after odor presentation, sometimes taking more than 100 s.

Criterion: single 144 s trials from five ants at 48%, M3 traces of 6G, and their mean. Return to
baseline as the first volume after the peak where the mean trace falls (a) below baseline mean plus 3
SD or (b) below 0.2; plateau as the time the mean trace stays at or above (c) 90% or (d) 80% of its
peak; odor presentation as (e) 3 s or (f) 8 s. Reproduced when, under one reading, both values are
within tolerance (30 +/- 3 s, 80 +/- 8 s) and at least one trial takes more than 100 s.

In [ ]:
t120 = np.arange(120) * al.VOLUME_S

def plateau_return(tr_, t_):
    pk = int(np.argmax(tr_))
    base = tr_[al.BASE]
    out = {}
    for name, frac in (("c", 0.9), ("d", 0.8)):
        above = tr_ >= frac * tr_[pk]
        a0 = pk
        while a0 > 0 and above[a0 - 1]:
            a0 -= 1
        a1 = pk
        while a1 < len(tr_) - 1 and above[a1 + 1]:
            a1 += 1
        out[name] = (a1 - a0 + 1) * al.VOLUME_S
    for name, bound in (("a", base.mean() + 3 * base.std(ddof=1)), ("b", 0.2)):
        below = np.where(tr_[pk:] < bound)[0]
        out[name] = t_[pk + below[0]] if len(below) else np.nan
    return out
res7 = []
for ident in IDS:
    long = tr[(tr.ident == ident) & (tr.nvol == 120) & (tr.glom == "6G")]
    print(ident, ": 144 s trials with 6G identified:", long.ant.nunique() // 1, "ants of", alarm[alarm.nvol == 120].ant.nunique(),
          "with a 144 s recording; trials", long[(long.roi == "disc") & (long.plane_reading == "a")].shape[0])
    for roi in ("disc", "region"):
        for rd in "ab":
            s = long[(long.plane_reading == rd) & (long.roi == roi)]
            if not len(s):
                continue
            t_ = t120 + float(s.t0.mean())
            mean = np.stack(s.trace).mean(0)
            pm = plateau_return(mean, t_)
            per = [plateau_return(x, t120 + t0) for x, t0 in zip(s.trace, s.t0)]
            for ret in "ab":
                for pl in "cd":
                    for on, t_on in (("e", 3.0), ("f", 8.0)):
                        plateau = pm[pl]
                        back = pm[ret] - t_on
                        per_back = [p[ret] - t_on for p in per]
                        res7.append({"ident": ident, "roi": roi, "plane": rd, "reading": ret + pl + on, "n trials": len(s), "plateau s": round(plateau, 1),
                                     "return s": round(back, 1), "per-trial return s": [round(x, 1) for x in per_back],
                                     "any > 100 s": bool(np.nanmax(per_back) > 100) if np.isfinite(per_back).any() else False,
                                     "within tolerance": bool(abs(plateau - 30) <= 3 and abs(back - 80) <= 8)})
            print(f"  {roi}, plane {rd}: mean 6G trace peak {mean.max():.3f} at {t_[mean.argmax()]:.1f} s; range of the mean trace {mean.min():.3f} to {mean.max():.3f}")
res7 = pd.DataFrame(res7)
res7["met"] = res7["within tolerance"] & res7["any > 100 s"]
print(res7.to_string(index=False))
print("readings meeting the criterion:", int(res7.met.sum()), "of", len(res7))
long = tr[(tr.nvol == 120) & (tr.glom == "6G") & (tr.roi == "disc")]
if len(long):
    fig, ax = plt.subplots(1, len(IDS), figsize=(5 * len(IDS), 3), squeeze=False)
    for k, ident in enumerate(IDS):
        s = long[(long.ident == ident) & (long.plane_reading == "a")]
        if not len(s):
            continue
        for x in s.trace:
            ax[0, k].plot(t120, x, lw=0.7)
        ax[0, k].plot(t120, np.stack(s.trace).mean(0), "k", lw=2)
        ax[0, k].axvspan(3, 8, color="0.9"); ax[0, k].set_title(f"6G, 6MHO 48%, 144 s, {ident}, disc, plane a", fontsize=8); ax[0, k].set_xlabel("s")
    plt.tight_layout(); plt.show()

**Result: not reproduced.** Under aiiy 6G is identified in all 5 ants with a 144 s recording (under bix in 1). With the disc the mean trace peaks at 0.252 to 0.281 at 33.1 s, stays at or above 90% of its peak for 2.4 to 3.6 s and at or above 80% for 4.8 to 8.4 s (expected about 30 s), and returns 27.5 to 56.6 s after odor presentation (expected 80 +/- 8 s). With the region ROI the return (70.8 to 91.5 s) is near the expected value but the plateau is 7.2 to 15.7 s. One trial returns more than 100 s after odor presentation (100.7 to 105.7 s). None of the 64 combinations of identification, ROI, plane and criterion reading meets the criterion.

## figs7c-temporal-lme

> Onset and peak times differ by pheromone and glomerulus; for PGb and PGa responses to heptanol and
> hexanol, onset time depends on pheromone, glomerulus and their interaction, and time to maximum on
> pheromone and glomerulus but not their interaction.

Criterion: M3 traces at 48%; onset and time to maximum (Methods); traces that never exceed the bound
excluded. Model per parameter: pheromone, glomerulus and their interaction, random effect for trial
nested in ant (lme). p-values read as (a) sequential tests in the order the Methods list (here
likelihood-ratio tests of nested ML fits, as statsmodels has no anova for mixed models), (b) marginal
tests (Wald z of sum-coded effects), (c) the coefficient tests of a treatment-coded summary. Reproduced
when, under every reading, the five effects have p < 0.05 and the time-to-maximum interaction has p
of 0.05 or more; partially reproduced when three or four of the five hold.

In [ ]:
lat = []
for _, r in tr[(tr.conc == 0.48) & (tr.nvol == 40) & tr.glom.isin(["PGb", "PGa"]) & tr.odor.isin(["heptanol", "hexanol"])].iterrows():
    on, mx = al.onset_and_max(r.trace, t40 + r.t0)
    lat.append({"ident": r.ident, "roi": r.roi, "ant": r.ant, "trial": f"{r.ant}-{r.trial}", "odor": r.odor, "glom": r.glom,
                "plane_reading": r.plane_reading, "onset": on, "tmax": mx})
lat = pd.DataFrame(lat)
print("traces per identification, ROI, plane reading and glomerulus:")
print(lat.groupby(["ident", "roi", "plane_reading", "glom"]).size().unstack().to_string())
print("excluded (never above the bound):")
print(lat.groupby(["ident", "roi", "plane_reading", "glom"]).onset.apply(lambda s: int(s.isna().sum())).unstack().to_string())
print(lat.dropna().groupby(["ident", "roi", "plane_reading", "glom", "odor"])[["onset", "tmax"]].agg(["mean", "std", "size"]).round(2).to_string())

def lme_p(d_, y):
    d_ = d_.dropna(subset=[y]).copy()
    vc = {"trial": "0 + C(trial)"}
    out = {}
    # (c) treatment coding, coefficient z-tests
    fc = smf.mixedlm(f"{y} ~ C(odor) * C(glom)", d_, groups=d_.ant, vc_formula=vc).fit(reml=True)
    out["converged"] = bool(fc.converged)
    pv = fc.pvalues
    out["c"] = {"pheromone": pv[[k for k in pv.index if k.startswith("C(odor)") and ":" not in k][0]],
                "glomerulus": pv[[k for k in pv.index if k.startswith("C(glom)") and ":" not in k][0]],
                "interaction": pv[[k for k in pv.index if ":" in k][0]]}
    # (b) sum coding, marginal z-tests
    fb = smf.mixedlm(f"{y} ~ C(odor, Sum) * C(glom, Sum)", d_, groups=d_.ant, vc_formula=vc).fit(reml=True)
    pv = fb.pvalues
    out["b"] = {"pheromone": pv[[k for k in pv.index if k.startswith("C(odor") and ":" not in k][0]],
                "glomerulus": pv[[k for k in pv.index if k.startswith("C(glom") and ":" not in k][0]],
                "interaction": pv[[k for k in pv.index if ":" in k][0]]}
    # (a) sequential likelihood-ratio tests (ML)
    forms = ["1", "C(odor)", "C(odor) + C(glom)", "C(odor) * C(glom)"]
    ll = [smf.mixedlm(f"{y} ~ {f}", d_, groups=d_.ant, vc_formula=vc).fit(reml=False).llf for f in forms]
    out["a"] = {k: float(stats.chi2.sf(2 * (ll[i + 1] - ll[i]), 1)) for i, k in enumerate(["pheromone", "glomerulus", "interaction"])}
    return out
rows, untestable = [], []
for (ident, roi, rd), d_ in lat.groupby(["ident", "roi", "plane_reading"]):
    for y in ("onset", "tmax"):
        levels = d_.dropna(subset=[y]).glom.nunique()
        if levels < 2:
            untestable.append(f"{ident}/{roi}/plane {rd}/{y}: only {sorted(d_.dropna(subset=[y]).glom.unique())} has traces above the bound, so the glomerulus factor has one level and the model cannot be fitted")
            continue
        try:
            ps = lme_p(d_, y)
        except Exception as e:  # noqa: BLE001
            untestable.append(f"{ident}/{roi}/plane {rd}/{y}: fit failed ({e})")
            continue
        conv = ps.pop("converged")
        for reading, p in ps.items():
            rows.append({"converged (c)": conv, **{"ident": ident, "roi": roi, "plane": rd, "p reading": reading, "parameter": y, **{k: round(float(v), 4) for k, v in p.items()}}})
for u in untestable:
    print("NOT TESTED:", u)
for ident in IDS:
    if not (lat.ident == ident).any() or (lat[lat.ident == ident].glom == "PGa").sum() == 0:
        print(f"NOT TESTED under {ident}: no PGa traces, so the claim cannot be tested under this identification")
lme = pd.DataFrame(rows, columns=["ident", "roi", "plane", "p reading", "parameter", "pheromone", "glomerulus", "interaction", "converged (c)"])
print(lme.to_string(index=False))
lme_j = []
for (ident, roi, rd, pr), g in lme.groupby(["ident", "roi", "plane", "p reading"]):
    if set(g.parameter) != {"onset", "tmax"}:
        continue
    on = g[g.parameter == "onset"].iloc[0]; mx = g[g.parameter == "tmax"].iloc[0]
    holds = [on.pheromone < 0.05, on.glomerulus < 0.05, on.interaction < 0.05, mx.pheromone < 0.05, mx.glomerulus < 0.05]
    lme_j.append({"ident": ident, "roi": roi, "plane": rd, "p reading": pr, "significant of 5": sum(holds),
                  "which": "".join("1" if h else "0" for h in holds), "tmax interaction p >= 0.05": bool(mx.interaction >= 0.05)})
lme_j = pd.DataFrame(lme_j)
print("\n'which' marks onset pheromone, glomerulus, interaction, time-to-maximum pheromone, glomerulus")
print(lme_j.to_string(index=False))

**Result: not reproduced, resting on the identification.** Under bix no PGa is identified, so the models cannot be fitted (the glomerulus factor has one level; listed as NOT TESTED above). Under aiiy, the identification that finds PGa in the most ants (4 of 13, 3 of them at 48%), the models fit with 18 PGa and 48 PGb traces at 48% (with the disc 8 or 9 PGa traces never exceed the bound and are excluded, with the region 2 or 3). Across the 12 combinations of ROI, plane and p-value reading, 1 to 4 of the 5 effects have p < 0.05, and the time-to-maximum interaction has p < 0.05 in 9, where the paper finds none (p = 0.88). The evidence is weak: PGa comes from 3 ants at 48%, against the paper's 8.

## Summary

| Claim | Outcome |
|---|---|
| fig1cd-new-pheromones-leave-nest | blocked (data_missing) |
| fig1e-categorical-response | blocked (data_missing) |
| fig1f-hexanol-like-native | blocked (data_missing) |
| fig1f-mho-differs | blocked (data_missing) |
| fig1f-vehicle-differs | blocked (data_missing) |
| table1-transgenesis | blocked (data_missing) |
| fig2b-s1a-transgene-fluorescence | blocked (data_missing) |
| fig2cd-gcamp-in-osns | blocked (other), elsewhere |
| figs1-expression-outside-al | blocked (other), elsewhere |
| figs2ab-glomerulus-counts | blocked (not_runnable), elsewhere |
| fig2e-t7-ammc | blocked (other), elsewhere |
| fig2f-ablation | blocked (other), elsewhere |
| ablation-al-volume-trend | blocked (not_runnable), elsewhere |
| figs2cf-gcamp-ants-behavior | blocked (data_missing) |
| figs3-single-insertion | blocked (not_runnable), elsewhere |
| fig3f-general-odorants-unique | not reproduced |
| fig3fg-hexanone-broad | reproduced |
| figs4a-bilateral-symmetry | partially reproduced |
| fig4b-vehicle-no-response | not reproduced |
| fig4-no-decreases | not reproduced |
| fig4-large-lasting-responses | not reproduced |
| figs5a-sparse-median-6 | reproduced (assumed) |
| figs5a-concentration-more-glomeruli | reproduced (assumed) |
| figs5b-shared-glomerulus | not reproduced |
| figs5c-mho-rarely-overlaps | reproduced (assumed) |
| fig5ab-focal-glomeruli | not reproduced |
| fig5c-s6-tuning | partially reproduced (assumed) |
| fig5c-6g-slow | partially reproduced (assumed) |
| figs7a-6g-extended | not reproduced |
| figs7c-temporal-lme | not reproduced (assumed) |
| fig5d-stereotyped-positions | not reproduced |
| fig6-shared-panic-pathway | not reproduced, imaging half |

"(assumed)" marks an outcome that rests on the adopted reading of a method the paper leaves open. The
paper drew its ROIs by eye. Of the eight M2 readings of the claims file, the outcomes of figs5a and
figs5c rest on the ROI's mean value (ImageJ's default measure) with the Methods' trial-mean rule, under
which both ROI drawings meet the criteria; the readings with the ROI's maximum do not. The
single-glomerulus claims rest on the focal glomeruli identified under bix (the adopted reading, which
finds PGb in 5 of 13 ants, 6G in 2 and PGa in none) or under aiiy (the reading that finds PGa in the
most ants, 4), and on the 15 um disc ROI; the claims file's other ROI, the region of at least 0.2,
takes in neighbouring glomeruli and is shown everywhere.

How the analysis differs from the paper's: Fiji's Image Stabilizer is replaced by whole-pixel phase
correlation on each plane, for the maps and for the single-glomerulus row bands; per-plane maps are
binned 4 x 4 (1.16 um); ROIs are drawn by rule inside an AL mask, not by eye, as regions of at least
0.2 that recur in two trials of an odor (drawing a keeps every such region, however small, as the
claims file has it; drawing b keeps those of glomerulus size); a recurrence is an overlap of at least
half the smaller region; the Methods' exclusion of trials with large motion artifacts has no rule, and
a variant leaves out the one trial where most of the AL is at least 0.2.

What was found about the dataset: it holds all three imaging experiments the paper describes (6
general odorant ants with the five analysed odorants, 3 bilateral ants, 13 alarm ants at the stated
concentration pairs, and five 144 s recordings), one file per plane and trial, 14,685 files, 1.26 TB
of uncompressed float64. No file or trial is deposited twice. The files give orientation, pixel size
and ages; the stimulus timing is only in free text. The planes' starting times step by 0.0335 s (1.105 s
per 33-plane volume), while the imaging rate field says 0.83 volumes/s (1.205 s) and the paper 27.5
frames/s (1.2 s per volume).

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Runtime of this notebook:

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")